# Chunk 29 — Cascade diagnostics (the gate for Chunks 30–34)

Reads the Chunk 28 caches and decides, before any GPU is spent on the cascade:

| cell | question | decision it fires |
|---|---|---|
| G | What are the baselines without 55×55? What does Pillar 3 look like at 35/45? Which functioning label? | baselines, Pillar 3 restatement, `LABEL_MIN` |
| A | Can a product of one-port resonators represent the **true** curves? | VIABLE / MARGINAL / DEAD, K, baseline order, synthesizer floor |
| B | How much of the measured MSE is frequency error vs depth error? | sizes the continuous-f0 gain |
| C | Does the training objective weight frequencies like the metric does? | quantifies the loss misalignment |
| D | What do clamping, mirror averaging and seed ensembling buy for free? | mirror augmentation yes/no |
| E | Are rescaled Laplacian eigenvalues grid-comparable and predictive? | spectral variant or DROPPED |
| F | Guards | — |

**Scope: grids 25, 35, 45 only.** No training, no optimizer, no model selection. The only forward passes are eval-mode passes of existing checkpoints in Cells D and E. Every file written starts with `ch29_`.

**Runtime:** run on a runtime with many CPUs; Cell A's fits are CPU-parallel and print a projection before they start.

## Cell 1 — Installs
Installs PyTorch Geometric (used only for the eval-mode forward passes in Cells D and E) and pyarrow for the parquet caches.

In [ ]:
!pip -q install torch_geometric pyarrow

## Cell 2 — Repository
Clones or pulls `asparagusD/antenna_gnn` and puts `src/` on the path so `AntennaGNN` can be imported in Cell D.

In [ ]:
import os, sys
REPO_DIR = '/content/antenna_gnn'
import subprocess
if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/asparagusD/antenna_gnn.git', REPO_DIR], check=True)
sys.path.insert(0, f'{REPO_DIR}/src')
print('repo ready:', REPO_DIR)

## Cell 3 — Drive
Mounts Google Drive and sets `DATA_ROOT` and `RAW_DATA`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA = '/content/drive/MyDrive/antenna_dataset'
print('DATA_ROOT:', DATA_ROOT)

## Setup — constants, caches, scope filter
**Inputs:** `ch28_per_sample.parquet`, `ch28_curves.npz`, `s11_mean.npy` / `s11_std.npy` (read only).

**What it does:**
- Defines `PIXEL_SIZE_MM` and the depth bins as `(lo, hi, label)` triples.
- Captures the mtime of every `ch28_*` artifact so Cell F can prove none was modified.
- Asserts 1,125,040 rows / 49 models before filtering, then drops every grid-55 row.
- Defines `curves_for()`, which aligns curve arrays to parquet rows. It proves alignment by checking that the parquet's `true_min_db` equals each curve's minimum.
- Defines the feed location from `alternate.m` V3.1.1 (MATLAB rounding).

**Writes:** nothing.

In [ ]:
# ── Setup: imports, constants, caches, scope filter, provenance helpers ──
import os, sys, json, glob, time, math, hashlib, warnings, importlib, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.interpolate import PchipInterpolator
from scipy.optimize import least_squares
from scipy.signal import find_peaks
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, r2_score

ART = f"{DATA_ROOT}/artifacts"
FIG = f"{DATA_ROOT}/figures"
os.makedirs(ART, exist_ok=True); os.makedirs(FIG, exist_ok=True)

GRIDS = (25, 35, 45)
PIXEL_SIZE_MM = {25: 1.296, 35: 0.925, 45: 0.720}
PATCH_SIDE_MM = 32.4
F_GHZ = np.linspace(1.0, 4.0, 201)
BIN_GHZ = F_GHZ[1] - F_GHZ[0]
DEPTH_BINS = [(-np.inf, -30.0, 'deep'), (-30.0, -20.0, 'mid'),
              (-20.0, -15.0, 'shallow'), (-15.0, -10.0, 'marginal')]

def depth_bin(v):
    for lo, hi, lbl in DEPTH_BINS:
        if lo < v <= hi:
            return lbl
    return 'nonfunc'

for g in GRIDS:
    assert abs(g * PIXEL_SIZE_MM[g] - PATCH_SIDE_MM) / PATCH_SIDE_MM < 0.01, \
        f"PIXEL_SIZE_MM[{g}] gives patch side {g*PIXEL_SIZE_MM[g]:.3f} mm, not ~32.4 mm"
print("Pixel-size table consistent with a 32.4 mm patch side (within 1%).")

# ── Artifact bookkeeping: every file this notebook writes goes through here ──
WRITTEN = []
def out_path(name, kind='art'):
    assert os.path.basename(name).startswith('ch29_'), f"refusing to write non-ch29 file {name}"
    p = f"{ART if kind == 'art' else FIG}/{name}"
    WRITTEN.append(p)
    return p

# ── Capture mtimes of every ch28_* artifact for Cell F ──
CH28_MTIMES = {p: os.path.getmtime(p) for p in glob.glob(f"{ART}/ch28_*")}
print(f"Captured mtimes of {len(CH28_MTIMES)} ch28_* artifacts.")

# ── Load the Chunk 28 caches ──
PARQUET = f"{ART}/ch28_per_sample.parquet"
CURVES = f"{ART}/ch28_curves.npz"
df_all = pd.read_parquet(PARQUET)
npz = np.load(CURVES)
N_ROWS_PRE, N_MODELS_PRE = len(df_all), df_all['model_id'].nunique()
print(f"Parquet: {N_ROWS_PRE:,} rows, {N_MODELS_PRE} models (before grid-55 filter)")
assert N_ROWS_PRE == 1_125_040, f"STOP: expected 1,125,040 rows, got {N_ROWS_PRE:,}"
assert N_MODELS_PRE == 49, f"STOP: expected 49 models, got {N_MODELS_PRE}"
assert 'best_model__25x25_test__true' in npz.files, "STOP: curves cache lacks best_model__25x25_test__true"
print("Cache assertions passed.")
print("Splits in parquet:", sorted(df_all['split'].unique()))
print("Grids in parquet:", sorted(df_all['grid_size'].unique()))
print(f"Curve arrays ({len(npz.files)}):")
for k in sorted(npz.files):
    print(f"   {k:<55} {npz[k].shape}")

# ── Scope filter: grids 25/35/45 only ──
ref_mid = df_all['model_id'].iloc[0]
_drop = df_all[(df_all['grid_size'] == 55) & (df_all['model_id'] == ref_mid)].groupby('split').size()
print("\nGrid-55 rows dropped per split (per model):")
print(_drop.to_string() if len(_drop) else "   none")
df = df_all[df_all['grid_size'].isin(GRIDS)].copy()
assert not (df['grid_size'] == 55).any()
df['label_ref'] = df['is_functioning'].astype(bool)
df['label_min'] = df['true_min_db'] < -10.0
df['pred_func'] = df['pred_min_db'] < -10.0
print(f"After filter: {len(df):,} rows")

# ── Curve access, aligned to parquet rows (npz rows follow parquet sample_idx order) ──
def curves_for(model_id, split, grids=GRIDS):
    kt, kp = f"{model_id}__{split}__true", f"{model_id}__{split}__pred"
    if kt not in npz.files or kp not in npz.files:
        return None
    rows = (df_all[(df_all['model_id'] == model_id) & (df_all['split'] == split)]
            .sort_values('sample_idx').reset_index(drop=True))
    T, P = npz[kt], npz[kp]
    assert len(rows) == len(T) == len(P), f"{kt}: {len(rows)} parquet rows vs {len(T)} curves"
    # alignment proof: parquet true_min_db must equal the curve minimum
    gap = np.abs(rows['true_min_db'].values - T.min(axis=1)).max()
    assert gap < 1e-3, f"{kt}: curve/parquet misalignment, max |dmin| = {gap:.4g}"
    keep = rows['grid_size'].isin(grids).values
    return rows[keep].reset_index(drop=True), T[keep].astype(np.float64), P[keep].astype(np.float64)

print("\nCurve sets available for best_model:",
      [s for s in sorted(df_all['split'].unique()) if curves_for('best_model', s) is not None])

# ── Graph files and the frozen builder (for Cells D and E) ──
def graph_path(grid, local_idx):
    sub = 'processed' if grid == 25 else 'processed_finetune'
    for p in (f"/content/local_graphs/{grid}x{grid}/sample_{local_idx}.pt",
              f"{DATA_ROOT}/data/{sub}/{grid}x{grid}/sample_{local_idx}.pt"):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"graph {grid}x{grid} sample_{local_idx}.pt not found")

def mround(x):
    """MATLAB round (half away from zero). Python round() is banker's rounding."""
    return int(math.floor(x + 0.5))

def feed_rc(N):
    """0-based (row, col) of the feed pixel per alternate.m V3.1.1."""
    return mround(0.25 * N) - 1, mround(N / 2) - 1

def graph_arrays(data):
    """Recover (N, pattern, seed_mask) from a stored graph: node order i*N+j, virtual node last."""
    x = data.x.cpu().numpy()
    assert x[-1, 3] == -1.0, "last node is not the virtual node (is_seed != -1)"
    n = x.shape[0] - 1
    N = int(round(math.sqrt(n)))
    assert N * N == n, f"{n} pixel nodes is not a square grid"
    return N, x[:-1, 0].reshape(N, N).astype(np.int8), x[:-1, 3].reshape(N, N).astype(np.int8)

# Checkpoint and normalization resolution (used only by Cells D and E)
CKPT = f"{DATA_ROOT}/checkpoints"
def find_s11_stats():
    for d in (DATA_ROOT, ART, f"{DATA_ROOT}/data", f"{DATA_ROOT}/stats", f"{DATA_ROOT}/data/processed"):
        if os.path.exists(f"{d}/s11_mean.npy") and os.path.exists(f"{d}/s11_std.npy"):
            return f"{d}/s11_mean.npy", f"{d}/s11_std.npy"
    hits = glob.glob(f"{DATA_ROOT}/**/s11_mean.npy", recursive=True)
    assert hits, "s11_mean.npy not found under DATA_ROOT"
    d = os.path.dirname(hits[0])
    return f"{d}/s11_mean.npy", f"{d}/s11_std.npy"
S11_MEAN_PATH, S11_STD_PATH = find_s11_stats()
S11_MEAN, S11_STD = np.load(S11_MEAN_PATH).astype(np.float64), np.load(S11_STD_PATH).astype(np.float64)
assert S11_MEAN.shape == (201,) and S11_STD.shape == (201,)
print(f"s11 stats: {S11_MEAN_PATH} (read only, never recomputed)")

## Setup — grid shape checks
Reads one raw `.mat` and one processed graph per grid. Asserts the grids are square (`Ny == Nx == grid`) and that each graph has `grid²` pixel nodes plus one virtual node. Also checks the graph's `pixel_size_mm` attribute, where present, against the table. Stops on any mismatch.

In [ ]:
# ── Grid shape checks: one raw .mat and one processed graph per grid ──
import scipy.io as sio, torch
for g in GRIDS:
    pats = [f"{RAW_DATA}/fine-tuning dataset/{g}x{g}/**/Mat_Files/*.mat",
            f"{RAW_DATA}/**/{g}x{g}/**/Mat_Files/*.mat", f"{RAW_DATA}/**/{g}x{g}/**/*.mat"]
    mat = next((p for pat in pats for p in glob.iglob(pat, recursive=True)), None)
    if mat is None:
        print(f"{g}x{g}: no raw .mat found under RAW_DATA; relying on the graph check below")
    else:
        Ny, Nx = sio.loadmat(mat, variable_names=['patch_pattern'])['patch_pattern'].shape
        print(f"{g}x{g}: {os.path.basename(mat)} patch_pattern shape ({Ny}, {Nx})")
        assert Ny == Nx == g, f"STOP: {mat} is {Ny}x{Nx}, expected square {g}x{g}"
    lidx = int(df[(df['grid_size'] == g) & (df['model_id'] == 'best_model')]['local_idx'].iloc[0])
    d = torch.load(graph_path(g, lidx), weights_only=False)
    n_pix = d.x.shape[0] - 1
    assert n_pix == g * g, f"STOP: graph sample_{lidx} has {n_pix} pixel nodes, expected {g*g}"
    if hasattr(d, 'pixel_size_mm'):
        ps = float(d.pixel_size_mm)
        print(f"   graph pixel_size_mm attribute {ps:.4f} vs table {PIXEL_SIZE_MM[g]:.4f}")
        assert abs(ps - PIXEL_SIZE_MM[g]) / PIXEL_SIZE_MM[g] < 0.01, "STOP: pixel-size table disagrees with graphs"
    print(f"   graph {g}x{g}: {n_pix} pixel nodes + 1 virtual node; grid*h = {g*PIXEL_SIZE_MM[g]:.3f} mm")

## Cell G1 — Baselines restated without 55×55
**What it does:** recomputes MSE, MAE, accuracy and AUROC for every model on val and test, under all three pooling schemes. It does this twice, with 55×55 (`with55`) and without (`no55`), using the stored label for comparability with Chunk 28. It then prints the per-grid and `POOL_EQUAL_GRID_TEST` baselines the 0.17 dB² target is measured against.

**Note:** Chunk 28 already excluded 55×55 from its headline pooled numbers, so differences here are against an all-grids recomputation. The training contamination of `ceiling_fullpool` (its pool contained 55×55) stands regardless.

**Writes:** `ch29_no55_restatement.csv`.

In [ ]:
# ── Cell G1: scope restatement without 55x55 ──
def metrics(sub, label='label_ref'):
    if len(sub) == 0:
        return dict(n=0, mse=np.nan, mae=np.nan, acc=np.nan, auroc=np.nan)
    y = sub[label].values
    auc = roc_auc_score(y, -sub['pred_min_db'].values) if 0 < y.sum() < len(y) else np.nan
    return dict(n=len(sub), mse=sub['sample_mse_db2'].mean(), mae=sub['sample_mae_db'].mean(),
                acc=(sub['pred_func'].values == y).mean(), auroc=auc)

SPLITS = set(df_all['split'].unique())
HAS_25_VAL = '25x25_val' in SPLITS
print("25x25 validation split in the cache:", HAS_25_VAL)

def scope_rows(d, model, role, grids):
    """role 'VAL' or 'TEST'. 25x25 comes from 25x25_<role>; 35/45(/55) from finetune_<role>."""
    parts = {}
    s25 = f"25x25_{role.lower()}"
    for g in grids:
        split = s25 if g == 25 else f"finetune_{role.lower()}"
        parts[g] = d[(d['model_id'] == model) & (d['split'] == split) & (d['grid_size'] == g)]
    return parts

def pooled(parts, label='label_ref'):
    out = {}
    for g, s in parts.items():
        out[f"grid_{g}"] = metrics(s, label)
    ft = [s for g, s in parts.items() if g != 25 and len(s)]
    out['POOL_FINETUNE'] = metrics(pd.concat(ft), label) if ft else metrics(pd.DataFrame(), label)
    nonempty = [s for s in parts.values() if len(s)]
    out['POOL_ALL_SAMPLES'] = metrics(pd.concat(nonempty), label) if nonempty else metrics(pd.DataFrame(), label)
    per = [out[f"grid_{g}"] for g in parts if out[f"grid_{g}"]['n'] > 0]
    if per:
        out['POOL_EQUAL_GRID'] = {k: (sum(m['n'] for m in per) if k == 'n' else np.nanmean([m[k] for m in per]))
                                  for k in ('n', 'mse', 'mae', 'acc', 'auroc')}
    else:
        out['POOL_EQUAL_GRID'] = metrics(pd.DataFrame(), label)
    return out

df_all['label_ref'] = df_all['is_functioning'].astype(bool)
df_all['label_min'] = df_all['true_min_db'] < -10.0
df_all['pred_func'] = df_all['pred_min_db'] < -10.0

rows = []
for model in sorted(df_all['model_id'].unique()):
    for role in ('VAL', 'TEST'):
        # if there is no 25x25_val split, grid_25 is simply absent from the VAL scopes;
        # it is never substituted with the test split
        for tag, grids, d in (('with55', (25, 35, 45, 55), df_all), ('no55', GRIDS, df)):
            res = pooled(scope_rows(d, model, role, grids))
            for scope, m in res.items():
                if m['n'] == 0:
                    continue
                rows.append(dict(model_id=model, role=role, variant=tag, scope=f"{scope}_{role}", **m))
rest = pd.DataFrame(rows)
rest.to_csv(out_path('ch29_no55_restatement.csv'), index=False)

for model in ('best_model', 'ceiling_fullpool'):
    sub = rest[rest['model_id'] == model]
    if sub.empty:
        print(f"!! {model} not in parquet"); continue
    piv = sub.pivot_table(index='scope', columns='variant', values=['mse', 'mae', 'acc', 'auroc'])
    print(f"\n=== {model}: old (with 55) vs new (no 55), label = stored is_functioning ===")
    print(piv.round(4).to_string())

print("\nNOTE: Chunk 28 already excluded 55x55 from its headline pooled numbers, so 'with55'")
print("      here is the all-grids recomputation, not necessarily what Chunk 28 printed.")
_ch28 = f"{ART}/ch28_mse_table.csv"
if os.path.exists(_ch28):
    print(f"      Compare against {_ch28} directly for the published values.")

print("\n=== BASELINES the 0.17 dB^2 target is measured against (MSE dB^2, no 55) ===")
for model in ('best_model', 'ceiling_fullpool'):
    sub = rest[(rest['model_id'] == model) & (rest['variant'] == 'no55') & (rest['role'] == 'TEST')]
    for sc in ('grid_25_TEST', 'grid_35_TEST', 'grid_45_TEST', 'POOL_EQUAL_GRID_TEST'):
        v = sub[sub['scope'] == sc]['mse']
        print(f"   {model:<18} {sc:<22} {v.iloc[0] if len(v) else float('nan'):.4f}")
print("\nceiling_fullpool was TRAINED on a pool containing 55x55. It is a reference only,")
print("never a reported model. No final model may descend from it.")

## Cell G2 — Pillar 3 restated at 35×35 and 45×45
**What it does:** reports `best_model`'s zero-shot performance (trained on 25×25 only) at node ratios 1.96× and 3.24×, under both labels. The old headline (0.657 dB at 4.84×) was a 55×55 number. Fine-tuned arms are listed and excluded, because their pools were drawn from a pool containing 55×55.

**Writes:** `ch29_pillar3_no55.csv`.

In [ ]:
# ── Cell G2: Pillar 3 restated at 35x35 and 45x45 (zero-shot best_model) ──
NODE_RATIO = {35: (35 / 25) ** 2, 45: (45 / 25) ** 2}
p3 = []
for role in ('val', 'test'):
    for g in (35, 45):
        sub = df[(df['model_id'] == 'best_model') & (df['split'] == f'finetune_{role}') & (df['grid_size'] == g)]
        for lbl in ('label_ref', 'label_min'):
            m = metrics(sub, lbl)
            p3.append(dict(model_id='best_model', training='25x25 only (zero-shot)', role=role, grid=g,
                           node_ratio=round(NODE_RATIO[g], 2), label=lbl, **m))
base25 = df[(df['model_id'] == 'best_model') & (df['split'] == '25x25_test')]
for lbl in ('label_ref', 'label_min'):
    p3.append(dict(model_id='best_model', training='25x25 only (in-distribution)', role='test', grid=25,
                   node_ratio=1.0, label=lbl, **metrics(base25, lbl)))
p3 = pd.DataFrame(p3)
print(p3.round(4).to_string(index=False))

print("\nData-efficiency arms: every fine-tuned checkpoint in the cache was trained on a")
print("pool drawn from the 35/45/55 fine-tune pool, so none can be used for a 55-free")
print("Pillar 3 claim without retraining. They are listed and EXCLUDED:")
_others = sorted(m for m in df['model_id'].unique() if m != 'best_model')
print("   " + ", ".join(_others))
p3.to_csv(out_path('ch29_pillar3_no55.csv'), index=False)

## Cell G3 — The functioning label
**Pre-registered:** `LABEL_MIN` (the true curve's minimum < −10 dB) is canonical.

**What it does:**
- Prints the confusion matrix between the stored label and `LABEL_MIN` per split and grid.
- Characterises the disagreements: how many have their minimum in an edge bin, and how many lie within 0.1 dB of −10.
- Restates every model's accuracy and AUROC under both labels.

**Writes:** `ch29_label_restatement.csv`.

In [ ]:
# ── Cell G3: is_functioning decision ──
print("PRE-REGISTERED: LABEL_MIN (min over 201 bins of the TRUE curve < -10 dB) is canonical,")
print("because the cascade classifies from the curve minimum, and edge-bin minima are real")
print("in-band matches that the stored findpeaks-based label can miss.\n")

one = df[df['model_id'] == 'best_model']
conf = []
for (split, g), sub in one.groupby(['split', 'grid_size']):
    r, m = sub['label_ref'].values, sub['label_min'].values
    dis = sub[r != m]
    edge = dis['true_argmin_bin'].isin([0, 200]).sum() if 'true_argmin_bin' in dis else np.nan
    near = (np.abs(dis['true_min_db'] + 10.0) <= 0.1).sum()
    conf.append(dict(split=split, grid=g, n=len(sub), ref1_min1=int((r & m).sum()),
                     ref1_min0=int((r & ~m).sum()), ref0_min1=int((~r & m).sum()),
                     ref0_min0=int((~r & ~m).sum()), disagree=len(dis),
                     disagree_edge_bin=int(edge), disagree_within_0p1dB=int(near)))
conf = pd.DataFrame(conf)
print(conf.to_string(index=False))

lab = []
for model in sorted(df['model_id'].unique()):
    for split, grids in (('25x25_test', (25,)), ('finetune_val', (35, 45)), ('finetune_test', (35, 45))):
        for g in grids:
            sub = df[(df['model_id'] == model) & (df['split'] == split) & (df['grid_size'] == g)]
            if len(sub) == 0:
                continue
            a, b = metrics(sub, 'label_ref'), metrics(sub, 'label_min')
            lab.append(dict(model_id=model, split=split, grid=g, n=a['n'],
                            acc_ref=a['acc'], acc_min=b['acc'], d_acc_pp=100 * (b['acc'] - a['acc']),
                            auroc_ref=a['auroc'], auroc_min=b['auroc']))
lab = pd.DataFrame(lab)
lab.to_csv(out_path('ch29_label_restatement.csv'), index=False)
print(f"\nAccuracy shift LABEL_REF -> LABEL_MIN across {lab['model_id'].nunique()} models:"
      f" median {lab['d_acc_pp'].median():+.2f} pp, range [{lab['d_acc_pp'].min():+.2f}, {lab['d_acc_pp'].max():+.2f}] pp")
print(lab[lab['model_id'].isin(['best_model', 'ceiling_fullpool'])].round(4).to_string(index=False))

LABEL = 'label_min'
DECISIONS = {'label': 'LABEL_MIN'}
print("\nDECISION: LABEL_MIN is canonical for Chunks 29-34 and the paper.")

## Cell A (1/3) — Resonator model and fitter
**What it does:**
- Defines the reference synthesizer, which Chunk 30's `src/synth.py` must match:
  - a product of one-port resonator factors in linear |Γ|, times a sigmoid polynomial baseline
  - a smooth floor `10·log10(ρ² + 1e-6)`
  - f0 bounded to [0.8, 4.2] GHz, so edge-bin minima can be out-of-band flanks
- The fitter uses a deepest-minima initialisation. Edge descents become out-of-band candidates, and a second start is tried when the global minimum is in an edge bin. Fitting uses `least_squares` against **floored** truth.
- Runs a unit test: a synthetic two-resonance curve must be recovered with f0 error < 1 MHz.

**Writes:** nothing.

In [ ]:
# ── Cell A (1/3): the resonator model and its fitter ──
# This synthesizer is the reference implementation; Chunk 30's src/synth.py must match it.
T_NORM = (F_GHZ - 2.5) / 1.5
LOGQ_LO, LOGQ_HI = np.log(3.0), np.log(500.0)
F0_LO, F0_HI = 0.8, 4.2
EPS2_60 = 1e-6   # smooth -60 dB floor

def synth_lin(theta, K, order):
    c = theta[:order + 1]
    p = theta[order + 1:].reshape(K, 4)          # f0, logQ, rmin, s
    poly = sum(c[i] * T_NORM ** i for i in range(order + 1))
    base = 1.0 / (1.0 + np.exp(-poly))
    rho = base.copy()
    for f0, lq, r, s in p:
        Q = np.exp(lq)
        x = 2.0 * Q * (F_GHZ - f0) / f0
        b = (1.0 - r) / (1.0 + r)
        A, B = (b - 1.0) ** 2, (b + 1.0) ** 2
        rho = rho * (1.0 - s * (1.0 - np.sqrt((A + x * x) / (B + x * x))))
    return rho, base

def to_db(rho, eps2=EPS2_60):
    return 10.0 * np.log10(rho ** 2 + eps2)

def floor_truth(y, eps2=EPS2_60):
    """Pass the truth through the SAME smooth floor as the model."""
    return 10.0 * np.log10(10.0 ** (y / 10.0) + eps2)

def _logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

def _q_from_width(yfl, idx):
    d = yfl[idx]; level = d / 2.0
    l = idx
    while l > 0 and yfl[l] < level: l -= 1
    r = idx
    while r < len(yfl) - 1 and yfl[r] < level: r += 1
    width = max(F_GHZ[r] - F_GHZ[l], BIN_GHZ)
    return float(np.clip(F_GHZ[idx] / width, 3.5, 450.0))

def _inits(yfl, K):
    """Deepest-minima initialisation. Edge descents (curve still falling at bin 0 or 200)
    are treated as candidate out-of-band resonances at 0.9 / 4.1 GHz; if the global
    minimum is in an edge bin, a second start with that resonator in-band is also tried."""
    lin = 10.0 ** (yfl / 20.0)
    base0 = float(np.clip(np.percentile(lin, 90), 0.05, 0.995))
    base_db = 20.0 * np.log10(base0)
    pk, _ = find_peaks(-yfl, prominence=0.3)
    cands = [(float(yfl[i]), int(i), float(F_GHZ[i])) for i in pk]
    if yfl[0] < yfl[1] and yfl[0] < base_db - 1.0:
        cands.append((float(yfl[0]), 0, 0.9))
    if yfl[-1] < yfl[-2] and yfl[-1] < base_db - 1.0:
        cands.append((float(yfl[-1]), 200, 4.1))
    cands.sort()
    real = cands[:K]
    def res_for(v, i, f0):
        rmin = float(np.clip(10 ** (v / 20.0) / base0, 2e-3, 0.99))
        if f0 < 1.0 or f0 > 4.0:          # out-of-band: the edge value is a flank, not the null
            rmin = 0.1
        return [float(np.clip(f0, F0_LO + 1e-3, F0_HI - 1e-3)), np.log(_q_from_width(yfl, i)), rmin, 0.95]
    res = [res_for(*c) for c in real]
    spare = [1.6, 2.5, 3.4, 2.0]
    while len(res) < K:
        res.append([spare[len(res) % 4], np.log(20.0), 0.5, 0.02])
    starts = [res]
    gmin = int(np.argmin(yfl))
    if gmin < 3 or gmin > 197:
        alt = [list(r) for r in res]
        for r in alt:
            if r[0] < 1.0 or r[0] > 4.0 or abs(r[0] - F_GHZ[gmin]) < 0.05:
                r[0] = float(F_GHZ[gmin]); r[2] = float(np.clip(10 ** (yfl[gmin] / 20.0) / base0, 2e-3, 0.99))
                break
        starts.append(alt)
    return base0, starts

def fit_curve(y_raw, K, order, eps2=EPS2_60):
    """Fit one curve. Returns dict with params and fit statistics vs floored AND raw truth."""
    t0 = time.perf_counter()
    yfl = floor_truth(y_raw, eps2)
    base0, starts = _inits(yfl, K)
    lo = [-30.0] * (order + 1) + [F0_LO, LOGQ_LO, 1e-3, 0.0] * K
    hi = [30.0] * (order + 1) + [F0_HI, LOGQ_HI, 0.999, 1.0] * K
    best = None
    for st in starts:
        x0 = np.array([_logit(base0)] + [0.0] * order + [v for r in st for v in r], dtype=float)
        x0 = np.clip(x0, np.array(lo) + 1e-6, np.array(hi) - 1e-6)
        fun = lambda th: to_db(synth_lin(th, K, order)[0], eps2) - yfl
        try:
            r = least_squares(fun, x0, bounds=(lo, hi), method='trf', x_scale='jac', max_nfev=1000)
        except Exception:
            continue
        if best is None or r.cost < best.cost:
            best = r
    if best is None:
        return dict(ok=False, time=time.perf_counter() - t0)
    rho, base = synth_lin(best.x, K, order)
    yhat = to_db(rho, eps2)
    e_fl = yhat - yfl
    e_raw = yhat - y_raw
    sub = y_raw < 10 * np.log10(eps2)
    p = best.x[order + 1:].reshape(K, 4)
    return dict(ok=True, converged=bool(best.status > 0), theta=best.x, yhat=yhat,
                mse_fl=float(np.mean(e_fl ** 2)), mse_raw=float(np.mean(e_raw ** 2)),
                sse_raw=float(np.sum(e_raw ** 2)), sse_raw_subfloor=float(np.sum(e_raw[sub] ** 2)),
                used_oob=bool(np.any(((p[:, 0] < 1.0) | (p[:, 0] > 4.0)) & (p[:, 3] >= 0.5))),
                n_starts=len(starts), time=time.perf_counter() - t0)

# ── Unit test on a synthetic curve with known parameters (must recover them) ──
_true = np.array([2.0, 0.0, 0.0,  3.10, np.log(25.0), 0.05, 1.0,  1.85, np.log(15.0), 0.30, 1.0])
_y = to_db(synth_lin(_true, 2, 2)[0])
_r = fit_curve(_y, 2, 2)
_p = _r['theta'][3:].reshape(2, 4)
_p = _p[np.argsort(-_p[:, 0])]
assert _r['mse_fl'] < 1e-6, f"synthetic self-fit failed: mse {_r['mse_fl']:.3g}"
assert abs(_p[0, 0] - 3.10) < 1e-3 and abs(_p[1, 0] - 1.85) < 1e-3, f"f0 not recovered: {_p[:, 0]}"
print(f"Synthetic recovery test passed: mse {_r['mse_fl']:.2e}, f0 {_p[:, 0].round(4)}, "
      f"{_r['time']*1e3:.0f} ms")
print("Magnitude-only degeneracy: |Gamma| is identical for coupling b and 1/b; the")
print("undercoupled branch (rmin in (0,1)) is taken by convention. Depth and Q are unaffected.")

## Cell A (2/3) — Sample, flag, fit
**What it does:**
- Samples 2,000 true curves from the 25×25 test split plus 1,000 each from the 35×35 and 45×45 fine-tune val (`seed 29`).
- Counts sub-floor truth points.
- Flags each curve REFINED or UNREFINED by re-running V3.1.1's coarse candidate pass on the curve. This is approximate.
- Fits K = 1–4 (quadratic baseline) and K = 4 (cubic baseline) in parallel. It times 50 fits first and prints a projection; above 2 h it drops K = 1 and the cubic variant, never the sample.

**Writes:** nothing yet.

In [ ]:
# ── Cell A (2/3): sample true curves, flag REFINED / UNREFINED, run the fits ──
import multiprocessing as mp
rng = np.random.RandomState(29)

samp_rows, samp_curves = [], []
plan = [('25x25_test', 25, 2000), ('finetune_val', 35, 1000), ('finetune_val', 45, 1000)]
for split, g, n in plan:
    got = curves_for('best_model', split, grids=(g,))
    assert got is not None, f"no cached TRUE curves for best_model/{split}"
    rows, T, _ = got
    assert len(rows) >= n, f"{split}/{g}: only {len(rows)} curves, need {n}"
    pick = np.sort(rng.choice(len(rows), n, replace=False))
    r = rows.iloc[pick].copy(); r['split'] = split
    samp_rows.append(r); samp_curves.append(T[pick])
S = pd.concat(samp_rows).reset_index(drop=True)
Y = np.concatenate(samp_curves)
assert not (S['grid_size'] == 55).any()
S['curve_id'] = np.arange(len(S))
S['label_min'] = Y.min(axis=1) < -10.0
S['depth_bin'] = [depth_bin(v) if v < -10 else 'nonfunc' for v in Y.min(axis=1)]
S['edge_min'] = np.isin(Y.argmin(axis=1), [0, 1, 2, 198, 199, 200])
print(f"Sampled {len(S)} TRUE curves:", S.groupby('grid_size').size().to_dict())

# Label-floor census
sub_pts = (Y < -60.0)
print(f"TRUE points below -60 dB: {sub_pts.mean()*100:.3f}% of points, "
      f"{sub_pts.any(axis=1).mean()*100:.2f}% of curves; min {Y.min():.1f} dB")

# REFINED / UNREFINED reconstruction of V3.1.1's coarse pass (APPROXIMATE: the curve
# is itself a pchip interpolant of the solver's points, so this re-samples an interpolant)
F_COARSE = np.linspace(1.0, 4.0, 31)
def refined_flag(y):
    lin = 10.0 ** (y / 20.0)
    yc = 20.0 * np.log10(np.clip(PchipInterpolator(F_GHZ, lin)(F_COARSE), 1e-10, None))
    pk, _ = find_peaks(-yc, prominence=0.5)
    pk = pk[-yc[pk] >= 2.0]
    f_min = F_GHZ[np.argmin(y)]
    return bool(len(pk) and np.min(np.abs(F_COARSE[pk] - f_min)) <= 0.150)
S['refined'] = [refined_flag(y) for y in Y]
print("REFINED/UNREFINED reconstruction is approximate (re-samples a pchip interpolant).")
print(f"UNREFINED share: {(~S['refined']).mean()*100:.1f}% overall; among LABEL_MIN functioning "
      f"{(~S.loc[S['label_min'], 'refined']).mean()*100:.1f}%")

VARIANTS = [('K1q', 1, 2), ('K2q', 2, 2), ('K3q', 3, 2), ('K4q', 4, 2), ('K4c', 4, 3)]

def _job(args):
    cid, vname, K, order = args
    r = fit_curve(Y[cid], K, order)
    r.pop('yhat', None)
    return cid, vname, K, order, r

# Timing probe and projection
n_proc = max(1, os.cpu_count() or 1)
probe = [(i, 'K2q', 2, 2) for i in range(50)]
t0 = time.perf_counter(); _ = [_job(a) for a in probe]; t_fit = (time.perf_counter() - t0) / 50
mult = {'K1q': 0.6, 'K2q': 1.0, 'K3q': 1.7, 'K4q': 2.6, 'K4c': 2.9}   # cost relative to K=2
proj_h = len(S) * sum(mult[v] for v, _, _ in VARIANTS) * t_fit / n_proc / 3600
print(f"Median-ish fit time (K=2): {t_fit*1e3:.0f} ms; {n_proc} processes; projected total {proj_h:.2f} h")
if proj_h > 2.0:
    VARIANTS = [v for v in VARIANTS if v[0] not in ('K1q', 'K4c')]
    proj_h = len(S) * sum(mult[v] for v, _, _ in VARIANTS) * t_fit / n_proc / 3600
    print(f"!! Projection > 2 h: dropped K=1 and the cubic variant (never the sample). New projection {proj_h:.2f} h")
    assert proj_h <= 2.0, "STOP: still > 2 h after cutting variants; use a runtime with more CPUs"

jobs = [(cid, v, K, o) for v, K, o in VARIANTS for cid in S['curve_id']]
t0 = time.perf_counter()
with mp.get_context('fork').Pool(n_proc) as pool:
    results = pool.map(_job, jobs, chunksize=16)
print(f"{len(jobs):,} fits in {(time.perf_counter()-t0)/60:.1f} min")

fit_rows, par_rows = [], []
for cid, v, K, order, r in results:
    meta = S.iloc[cid]
    base = dict(curve_id=cid, split=meta['split'], grid=int(meta['grid_size']), local_idx=int(meta['local_idx']),
                variant=v, K=K, baseline_order=order)
    if not r['ok']:
        fit_rows.append(dict(**base, ok=False)); continue
    fit_rows.append(dict(**base, ok=True, converged=r['converged'], mse_fl=r['mse_fl'], mse_raw=r['mse_raw'],
                         sse_raw=r['sse_raw'], sse_raw_subfloor=r['sse_raw_subfloor'],
                         used_oob=r['used_oob'], time_s=r['time']))
    th = r['theta']; c = th[:order + 1]; p = th[order + 1:].reshape(K, 4)
    rho, bse = synth_lin(th, K, order)
    for k in range(K):
        f0, lq, rm, s = p[k]
        b_at = float(np.interp(np.clip(f0, 1, 4), F_GHZ, bse))
        par_rows.append(dict(**base, k=k, f0_GHz=f0, Q=float(np.exp(lq)), rmin=rm, s=s,
                             min_db_fit=float(20 * np.log10(max(rm * b_at, 1e-6))),
                             baseline_coeffs=json.dumps([float(x) for x in c]),
                             fit_mse=r['mse_fl'], converged=r['converged']))
FIT = pd.DataFrame(fit_rows).merge(S[['curve_id', 'label_min', 'depth_bin', 'edge_min', 'refined']], on='curve_id')
PAR = pd.DataFrame(par_rows)
print(f"Failed fits: {(~FIT['ok']).sum()}; non-converged: {(FIT['ok'] & ~FIT['converged'].fillna(False).astype(bool)).sum()}")

## Cell A (3/3) — Report and the pre-registered gate
**What it does:**
- Tabulates the fit MSE against floored and raw truth, by grid, label, depth bin, edge minimum and REFINED status.
- Decides the synthesizer floor: −80 dB if sub-floor points carry more than 10% of the raw residual, otherwise −60 dB.
- Prints the gate rule, then the verdict on all curves and on REFINED curves. It acts on REFINED if the two disagree.
- Cross-checks the deepest fitted f0 against the true minimum.

**Gate:**

| verdict | condition | consequence |
|---|---|---|
| VIABLE | median ≤ 0.03 and ≥ 90% < 0.10 dB² | K = 3, quadratic baseline, refiner deferred |
| MARGINAL | median ≤ 0.10 dB² | K = 4, cubic baseline, refiner active |
| DEAD | otherwise | Chunks 31–33 are not built |

**Writes:** `ch29_representability.csv`, `ch29_resonator_params_sample.parquet` (Chunk 30 validates its GPU fitter against it), `ch29_fit_stats_sample.parquet`, `figures/ch29_resonator_fits.png`.

In [ ]:
# ── Cell A (3/3): report, pre-registered gate, artifacts ──
def summ(sub, col):
    v = sub[col].dropna().values
    if len(v) == 0:
        return {}
    return {'n': len(v), 'median': np.median(v), 'p90': np.percentile(v, 90), 'max': v.max(),
            **{f'<{t}': (v < t).mean() for t in (0.01, 0.05, 0.10, 0.17)}}

rep = []
okF = FIT[FIT['ok']]
for v in okF['variant'].unique():
    fv = okF[okF['variant'] == v]
    groups = [('all', fv)] + [(f'grid_{g}', fv[fv['grid'] == g]) for g in GRIDS]
    groups += [('functioning', fv[fv['label_min']]), ('nonfunctioning', fv[~fv['label_min']])]
    groups += [(f'bin_{lbl}', fv[fv['depth_bin'] == lbl]) for _, _, lbl in DEPTH_BINS]
    groups += [('edge_min', fv[fv['edge_min']]), ('interior_min', fv[~fv['edge_min']])]
    groups += [('REFINED', fv[fv['refined']]), ('UNREFINED', fv[~fv['refined']])]
    for gname, sub in groups:
        for col, truth in (('mse_fl', 'floored'), ('mse_raw', 'raw')):
            st = summ(sub, col)
            if st:
                rep.append(dict(variant=v, group=gname, truth=truth, **st,
                                share_oob=sub['used_oob'].mean(), median_time_s=sub['time_s'].median()))
REP = pd.DataFrame(rep)
REP.to_csv(out_path('ch29_representability.csv'), index=False)
pd.set_option('display.width', 200)
print(REP[(REP['truth'] == 'floored') & REP['group'].isin(['all', 'grid_25', 'grid_35', 'grid_45',
      'functioning', 'edge_min', 'REFINED', 'UNREFINED'])].round(4).to_string(index=False))

# Label-floor share at the gate variant
gv = okF[okF['variant'] == 'K2q']
subfloor_share = gv['sse_raw_subfloor'].sum() / gv['sse_raw'].sum()
print(f"\nSub-floor (-60 dB) truth points carry {subfloor_share*100:.1f}% of the RAW K=2 residual.")
SYNTH_FLOOR_DB = -80 if subfloor_share > 0.10 else -60
print(f"Synthesizer floor for Chunks 30+: {SYNTH_FLOOR_DB} dB "
      f"({'1e-4' if SYNTH_FLOOR_DB == -80 else '1e-3'} in linear |Gamma|)")

print("""
PRE-REGISTERED GATE (floored truth, K = 2, pooled over grids 25/35/45; also on REFINED only;
if they disagree, act on REFINED and report the disagreement as a label-quality finding):
  VIABLE    median <= 0.03 dB^2 AND >= 90% of curves below 0.10 dB^2
            -> K = 3 slots, quadratic baseline, residual refiner DEFERRED
  MARGINAL  median <= 0.10 dB^2 but the 90% criterion fails
            -> K = 4, cubic baseline, residual refiner ACTIVE in Chunk 33
  DEAD      median > 0.10 dB^2
            -> do NOT build Chunks 31-33
""")
def branch(sub):
    v = sub['mse_fl'].values
    med, share = np.median(v), (v < 0.10).mean()
    b = 'VIABLE' if (med <= 0.03 and share >= 0.90) else ('MARGINAL' if med <= 0.10 else 'DEAD')
    return b, med, share
b_all, m_all, s_all = branch(gv)
b_ref, m_ref, s_ref = branch(gv[gv['refined']])
print(f"  all curves : median {m_all:.4f} dB^2, {s_all*100:.1f}% < 0.10  -> {b_all}")
print(f"  REFINED    : median {m_ref:.4f} dB^2, {s_ref*100:.1f}% < 0.10  -> {b_ref}")
BRANCH = b_ref
if b_all != b_ref:
    print(f"  LABEL-QUALITY FINDING: branches disagree ({b_all} vs {b_ref}); acting on REFINED.")
CFG = {'VIABLE': (3, 2, 'DEFERRED'), 'MARGINAL': (4, 3, 'ACTIVE'), 'DEAD': (None, None, 'n/a')}[BRANCH]
GATE_K, GATE_ORDER, REFINER = CFG
DECISIONS.update(cell_a_branch=BRANCH, K_slots=GATE_K, baseline_order=GATE_ORDER,
                 residual_refiner=REFINER, synth_floor_db=SYNTH_FLOOR_DB,
                 gate_median_all=m_all, gate_share_all=s_all, gate_median_refined=m_ref, gate_share_refined=s_ref,
                 subfloor_share=subfloor_share)
print(f"\nBRANCH: {BRANCH} | K slots: {GATE_K} | baseline order: {GATE_ORDER} | "
      f"refiner: {REFINER} | floor: {SYNTH_FLOOR_DB} dB")
print("NEXT NOTEBOOK:", "chunk30_backbone_targets_cache.ipynb" if BRANCH != 'DEAD'
      else "none of 31-33; write up MSE with the Gupta caveats, lead on MAE / invariance / cross-scale")

# Cross-check fitted f0 against the deepest true minimum (the .mat resonant_freqs is not in the
# Chunk 28 cache; the parquet's true_res_freq is the closest available proxy)
if BRANCH != 'DEAD':
    gvar = {(3, 2): 'K3q', (4, 3): 'K4c'}[(GATE_K, GATE_ORDER)]
    if gvar not in PAR['variant'].unique():
        gvar = 'K4q'
    act = PAR[(PAR['variant'] == gvar) & (PAR['s'] >= 0.5)]
    deep = act.loc[act.groupby('curve_id')['min_db_fit'].idxmin()].set_index('curve_id')
    func_ids = S[S['label_min']]['curve_id']
    fmin = pd.Series(F_GHZ[Y.argmin(axis=1)], index=S['curve_id'])
    d = (deep.reindex(func_ids)['f0_GHz'] - fmin.reindex(func_ids)).abs()
    print(f"Deepest active fitted f0 within 1 bin of the true argmin: {(d <= BIN_GHZ).mean()*100:.1f}% "
          f"of functioning curves (in-band argmin; out-of-band f0 legitimately differ)")
    if 'true_res_freq' in S.columns:
        rf = S.set_index('curve_id').reindex(func_ids)['true_res_freq']
        dd = (deep.reindex(func_ids)['f0_GHz'] - rf).abs().dropna()
        print(f"Within 1 bin of true_res_freq (findpeaks-based): {(dd <= BIN_GHZ).mean()*100:.1f}% of {len(dd)}")
        worst = dd.sort_values(ascending=False).head(20)
        print("20 worst |f0 - true_res_freq| (GHz):", worst.round(3).to_dict())

PAR.to_parquet(out_path('ch29_resonator_params_sample.parquet'), index=False)
FIT.to_parquet(out_path('ch29_fit_stats_sample.parquet'), index=False)

# Figure: 2 per depth bin + 2 edge-bin curves at the gate's K (K=2 if DEAD)
fig_var = {'VIABLE': 'K3q', 'MARGINAL': 'K4c', 'DEAD': 'K2q'}[BRANCH]
if fig_var not in okF['variant'].unique():
    fig_var = 'K2q'
fK, fO = {'K2q': (2, 2), 'K3q': (3, 2), 'K4q': (4, 2), 'K4c': (4, 3)}[fig_var]
ex = []
for _, _, lbl in DEPTH_BINS:
    ex += list(S[S['depth_bin'] == lbl]['curve_id'].head(2))
ex += list(S[S['edge_min'] & S['label_min']]['curve_id'].head(2))
fig, axs = plt.subplots(2, 5, figsize=(20, 7))
for ax, cid in zip(axs.ravel(), ex):
    r = fit_curve(Y[cid], fK, fO)
    m = FIT[(FIT['curve_id'] == cid) & (FIT['variant'] == fig_var)]
    ax.plot(F_GHZ, Y[cid], 'k-', lw=1.2, label='true')
    ax.plot(F_GHZ, r['yhat'], 'r--', lw=1.2, label=f'fit {fig_var}')
    ax.set_title(f"{S.iloc[cid]['grid_size']}x{S.iloc[cid]['grid_size']} {S.iloc[cid]['depth_bin']}"
                 f"{' edge' if S.iloc[cid]['edge_min'] else ''}  mse {r['mse_fl']:.3f}", fontsize=9)
    ax.set_xlabel('GHz'); ax.set_ylabel('S11 dB'); ax.grid(alpha=0.3)
axs[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(out_path('ch29_resonator_fits.png', 'fig'), dpi=130); plt.show()

## Cell B — MSE decomposition
**What it does:** for `best_model` on 25×25 test and on zero-shot 35×35 and 45×45 val, compares ACTUAL MSE with three counterfactuals:
- **FREQUENCY-ONLY:** the true curve shifted by the antenna's argmin error (pchip on linear |Γ|).
- **DEPTH-ONLY:** the true dip rescaled in linear |Γ| to the predicted minimum, keeping f0, width and the off-resonance level.
- **REMAINDER:** what is left after the two. These are not orthogonal, so the remainder can be negative.

Non-functioning curves are reported separately. The cell then verifies or refutes the ~0.38 dB² frequency share at 25×25.

**Writes:** `ch29_mse_decomposition.csv`, `figures/ch29_mse_decomposition.png`.

In [ ]:
# ── Cell B: decompose the measured MSE into frequency, depth and remainder ──
def shift_curve(y, df_ghz):
    """True curve shifted by df (GHz): value at f is y(f - df). pchip on linear |Gamma|, edges held."""
    lin = 10.0 ** (y / 20.0)
    f_src = np.clip(F_GHZ - df_ghz, F_GHZ[0], F_GHZ[-1])
    return 20.0 * np.log10(np.clip(PchipInterpolator(F_GHZ, lin)(f_src), 1e-10, None))

def depth_scale(y, target_min_db):
    """Rescale the dip in linear |Gamma| so its minimum hits the target, keeping f0, width and the
    off-resonance level: rho' = 1 - (1 - rho) * (1 - rho_t) / (1 - rho_min)."""
    lin = 10.0 ** (y / 20.0)
    rmin, rt = lin.min(), 10.0 ** (target_min_db / 20.0)
    if 1.0 - rmin < 1e-9:
        return y.copy()
    out = 1.0 - (1.0 - lin) * (1.0 - rt) / (1.0 - rmin)
    return 20.0 * np.log10(np.clip(out, 1e-10, None))

dec = []
for split, grids, tag in (('25x25_test', (25,), 'grid_25_test'), ('finetune_val', (35,), 'grid_35_val'),
                          ('finetune_val', (45,), 'grid_45_val')):
    rows, T, P = curves_for('best_model', split, grids=grids)
    for i in range(len(T)):
        t, p = T[i], P[i]
        func = t.min() < -10.0
        actual = np.mean((p - t) ** 2)
        dfreq = F_GHZ[np.argmin(p)] - F_GHZ[np.argmin(t)]
        fo = np.mean((shift_curve(t, dfreq) - t) ** 2)
        do = np.mean((depth_scale(t, p.min()) - t) ** 2)
        dec.append(dict(scope=tag, grid=grids[0], functioning=func,
                        depth_bin=depth_bin(t.min()) if func else 'nonfunc',
                        actual=actual, freq_only=fo, depth_only=do, remainder=actual - fo - do,
                        freq_err_bins=dfreq / BIN_GHZ))
DEC = pd.DataFrame(dec)
print("Caveat: frequency-only and depth-only contributions are NOT orthogonal; they need not sum")
print("to ACTUAL, and the remainder can be negative. Argmin is ill-defined for non-functioning")
print("curves, which are therefore reported separately.\n")
agg = (DEC.groupby(['scope', 'depth_bin'])[['actual', 'freq_only', 'depth_only', 'remainder']]
       .mean().join(DEC.groupby(['scope', 'depth_bin']).size().rename('n')))
pool = DEC.groupby('scope')[['actual', 'freq_only', 'depth_only', 'remainder']].mean()
pool['depth_bin'] = 'POOLED_all'; pool = pool.set_index('depth_bin', append=True)
fpool = DEC[DEC['functioning']].groupby('scope')[['actual', 'freq_only', 'depth_only', 'remainder']].mean()
fpool['depth_bin'] = 'POOLED_functioning'; fpool = fpool.set_index('depth_bin', append=True)
DEC_T = pd.concat([agg, pool, fpool]).sort_index()
print(DEC_T.round(4).to_string())
DEC_T.reset_index().to_csv(out_path('ch29_mse_decomposition.csv'), index=False)

fo25 = DEC[DEC['scope'] == 'grid_25_test']['freq_only'].mean()
a25 = DEC[DEC['scope'] == 'grid_25_test']['actual'].mean()
print(f"\nPromptbook estimate: frequency error ~0.38 of 1.32 dB^2 at 25x25.")
print(f"Measured: frequency-only {fo25:.3f} of {a25:.3f} dB^2 ({100*fo25/a25:.0f}%) -> "
      f"{'CONSISTENT' if abs(fo25 - 0.38) < 0.15 else 'NOT CONSISTENT'} with the estimate (|diff| < 0.15)")
print(f"Median |argmin frequency error| at 25x25 (functioning): "
      f"{DEC[(DEC['scope']=='grid_25_test') & DEC['functioning']]['freq_err_bins'].abs().median():.2f} bins")

fig, ax = plt.subplots(figsize=(12, 4.5))
lab = [f"{s}\n{b}" for s, b in DEC_T.index]
x = np.arange(len(DEC_T))
ax.bar(x, DEC_T['freq_only'], label='frequency-only')
ax.bar(x, DEC_T['depth_only'], bottom=DEC_T['freq_only'], label='depth-only')
ax.bar(x, DEC_T['remainder'].clip(lower=0), bottom=DEC_T['freq_only'] + DEC_T['depth_only'], label='remainder (>=0 part)')
ax.plot(x, DEC_T['actual'], 'k_', ms=18, mew=2, label='actual')
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=7, rotation=90); ax.set_ylabel('MSE dB$^2$')
ax.set_yscale('symlog', linthresh=0.1); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(out_path('ch29_mse_decomposition.png', 'fig'), dpi=130); plt.show()

## Cell C — Loss-space misalignment
**What it does:** on `best_model` 25×25 test, compares each frequency bin's share of the normalized training loss `((pred−true)/s11_std)²` with its share of the reported dB² metric. It reports the Spearman ρ between the two share vectors.

**Writes:** `ch29_loss_space.csv`, `figures/ch29_loss_space.png`.

In [ ]:
# ── Cell C: loss-space misalignment ──
v = S11_STD ** 2
print(f"s11_std per bin: min {S11_STD.min():.4f}, max {S11_STD.max():.4f}")
print(f"mean(s11_std^2) = {v.mean():.4f};  max/min(s11_std^2) = {v.max()/v.min():.1f}")
rows, T, P = curves_for('best_model', '25x25_test', grids=(25,))
E = P - T
L_norm_bin = np.mean((E / S11_STD) ** 2, axis=0)
L_db_bin = np.mean(E ** 2, axis=0)
c_norm, c_db = L_norm_bin / L_norm_bin.sum(), L_db_bin / L_db_bin.sum()
rho, pv = spearmanr(c_norm, c_db)
print(f"L_norm = {L_norm_bin.mean():.4f} (normalized units)   L_db = {L_db_bin.mean():.4f} dB^2")
print(f"Spearman rho between per-bin contribution vectors: {rho:+.3f} (p = {pv:.2g})")
print("A low or negative rho means the training objective and the reported metric disagree "
      "about which frequencies matter.")
top_n = np.argsort(-c_norm)[:10]; top_d = np.argsort(-c_db)[:10]
print(f"Top-10 bins by L_norm: {F_GHZ[np.sort(top_n)].round(3)}")
print(f"Top-10 bins by L_db  : {F_GHZ[np.sort(top_d)].round(3)}")
pd.DataFrame(dict(f_GHz=F_GHZ, s11_std=S11_STD, L_norm_bin=L_norm_bin, L_db_bin=L_db_bin,
                  share_norm=c_norm, share_db=c_db)).to_csv(out_path('ch29_loss_space.csv'), index=False)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(F_GHZ, c_norm, label='share of L_norm (training objective)')
ax.plot(F_GHZ, c_db, label='share of L_db (reported metric)')
ax.set_xlabel('GHz'); ax.set_ylabel('per-bin share'); ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"best_model, 25x25 test; Spearman rho = {rho:+.3f}")
plt.tight_layout(); plt.savefig(out_path('ch29_loss_space.png', 'fig'), dpi=130); plt.show()
DECISIONS['loss_space_spearman'] = float(rho)

## Cell D — Model utilities
**What it does:**
- Imports `AntennaGNN` from the repo and loads checkpoints, asserting 587,001 parameters.
- Defines eval-mode prediction under `torch.no_grad()`, with an optional hook on the 256-d input to `output_mlp`.
- Copies the frozen `build_pyg_graph` verbatim.
- Loads `best_model.pt`.

TF32 is disabled, and no optimizer exists anywhere in this notebook.

In [ ]:
# ── Model utilities for Cells D and E (eval-mode forward passes of EXISTING checkpoints only) ──
import torch
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

def resolve_antennagnn():
    for mod in ('model', 'models', 'gnn', 'antenna_gnn', 'models.gnn', 'model.gnn', 'antenna_gnn.model'):
        try:
            m = importlib.import_module(mod)
            if hasattr(m, 'AntennaGNN'):
                return m.AntennaGNN, mod
        except Exception:
            pass
    src = f"{REPO_DIR}/src"
    for py in sorted(glob.glob(f"{src}/**/*.py", recursive=True)):
        if 'class AntennaGNN' in open(py, errors='ignore').read():
            mod = os.path.relpath(py, src)[:-3].replace(os.sep, '.')
            m = importlib.import_module(mod)
            return m.AntennaGNN, mod
    raise ImportError("class AntennaGNN not found under src/")
AntennaGNN, _gnn_mod = resolve_antennagnn()
print(f"AntennaGNN imported from module '{_gnn_mod}'")

def load_gnn(path):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    sd = ck
    if isinstance(ck, dict):
        for key in ('model_state', 'model_state_dict', 'state_dict'):
            if key in ck:
                sd = ck[key]; break
    model = AntennaGNN(hidden_dim=128, heads=8, edge_dim=16, num_blocks=4, output_dim=201)
    model.load_state_dict(sd)
    n = sum(p.numel() for p in model.parameters())
    assert n == 587_001, f"{path}: {n} parameters, expected 587,001"
    return model.to(DEVICE).eval(), (ck if isinstance(ck, dict) else {})

_MEAN_T = torch.tensor(S11_MEAN, dtype=torch.float32, device=DEVICE)
_STD_T = torch.tensor(S11_STD, dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def predict_db(model, graphs, want_readout=False):
    """Returns predicted dB (n, 201) and optionally the 256-d input to output_mlp."""
    feats = []
    h = None
    if want_readout:
        assert hasattr(model, 'output_mlp'), "model has no output_mlp to hook"
        h = model.output_mlp.register_forward_pre_hook(lambda m, inp: feats.append(inp[0].detach().cpu().numpy()))
    out = []
    try:
        for b in DataLoader(graphs, batch_size=128, shuffle=False, num_workers=0):
            b = b.to(DEVICE)
            o = model(b)
            o = o[0] if isinstance(o, (tuple, list)) else o
            out.append((o * _STD_T + _MEAN_T).cpu().numpy())
    finally:
        if h is not None:
            h.remove()
    P = np.concatenate(out).astype(np.float64)
    return (P, np.concatenate(feats)) if want_readout else P

def load_graphs(grid, local_idxs):
    gs = []
    for i in local_idxs:
        d = torch.load(graph_path(grid, int(i)), weights_only=False)
        gs.append(Data(x=d.x, edge_index=d.edge_index, edge_attr=d.edge_attr, y=d.y))
    return gs

# frozen — validated in Chunk 12 (copied verbatim from the Chunk 5 builder)
def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)

    node_feats = []
    for i in range(N):
        for j in range(N):
            metal   = float(patch_pattern[i, j])
            x_norm  = j / (N - 1)
            y_norm  = i / (N - 1)
            is_seed = float(seed_mask[i, j])
            dist_f  = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])
    # virtual global node — is_seed = -1 is the load-bearing flag
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float)

    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx  = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0), (0,-1,1), (-1,0,2), (1,0,3)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    etype = etype_map[(m_ij, int(patch_pattern[ni, nj]))]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])

    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)
    y = torch.tensor(s11_db, dtype=torch.float).unsqueeze(0)   # RAW dB on disk
    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)

def same_graph(a, b):
    return (a.x.shape == b.x.shape and torch.allclose(a.x, b.x, atol=1e-6)
            and torch.equal(a.edge_index, b.edge_index) and torch.allclose(a.edge_attr, b.edge_attr))

BEST, _ = load_gnn(f"{CKPT}/best_model.pt")
print("best_model.pt loaded: 587,001 parameters")

## Cell D1 — Clamping
Measures the MSE reduction from `min(pred, 0)` per grid. A passive antenna cannot exceed 0 dB.

In [ ]:
# ── Cell D1: clamping to <= 0 dB ──
fw = []
for split, g in (('25x25_test', 25), ('finetune_val', 35), ('finetune_val', 45)):
    rows, T, P = curves_for('best_model', split, grids=(g,))
    m0, m1 = np.mean((P - T) ** 2), np.mean((np.minimum(P, 0.0) - T) ** 2)
    fw.append(dict(item='clamp', grid=g, split=split, mse_before=m0, mse_after=m1,
                   reduction=m0 - m1, reduction_pct=100 * (m0 - m1) / m0, share_points_above_0=(P > 0).mean()))
FW = pd.DataFrame(fw)
print(FW.round(4).to_string(index=False))

## Cell D2 — Mirror symmetry (column flip)
**Checks first:**
- The grid is odd and the feed is in the centre column.
- The feed pixel is metal in all 50 sampled graphs, which proves the feed location and orientation.
- The flipped feed maps to itself.
- The frozen builder reproduces a stored graph exactly.

**Then, for 200 antennas per grid:** flips the pattern and seed mask, rebuilds the graph through the builder, and predicts with `best_model`. Re-predictions must match the Chunk 28 cache (< 1e-3 dB). It reports the original-vs-mirror disagreement and the gain from averaging, split by whether the mirror leaves a hole in the forced centre block (out of the generator's support).

**Pre-registered:** an in-support averaging gain above 3% of MSE turns on mirror augmentation in Chunk 33.

In [ ]:
# ── Cell D2: mirror symmetry (column flip only) ──
print("A row flip is never valid (feed at row round(0.25*Ny)); only the column flip x -> -x is tested.")
rngD = np.random.RandomState(292)
mir_rows, MIRROR_OK = [], True
for split, g in (('25x25_test', 25), ('finetune_val', 35), ('finetune_val', 45)):
    rows, T, P = curves_for('best_model', split, grids=(g,))
    r0, c0 = feed_rc(g)
    # (i) Nx odd, feed column is the centre column, and the feed pixel is metal in every graph
    assert g % 2 == 1 and c0 == (g - 1) // 2, f"{g}: feed column {c0} is not the centre column"
    chk = load_graphs(g, rows['local_idx'].values[:50])
    pats = [graph_arrays(d) for d in chk]
    assert all(N == g for N, _, _ in pats), f"{g}: graph node count does not match the grid"
    feed_metal = np.mean([p[r0, c0] == 1 for _, p, _ in pats])
    alt_metal = np.mean([p[g - 1 - r0, c0] == 1 for _, p, _ in pats])
    print(f"{g}x{g}: feed (row {r0}, col {c0}) metal in {feed_metal*100:.0f}% of 50 graphs "
          f"(vertically flipped position: {alt_metal*100:.0f}%)")
    assert feed_metal == 1.0, f"{g}: feed pixel is not always metal - feed location or orientation is wrong"
    # (ii) the flipped feed node maps to itself
    assert g - 1 - c0 == c0
    # builder validation: rebuilding a stored graph from its own pattern must reproduce it exactly
    N, pat, seed = pats[0]
    if not same_graph(build_pyg_graph(pat, chk[0].y.squeeze(0).numpy(), seed, N), chk[0]):
        print(f"!! {g}x{g}: frozen builder does not reproduce the stored graph; mirror test SKIPPED")
        MIRROR_OK = False; continue
    pick = np.sort(rngD.choice(len(rows), 200, replace=False))
    orig = load_graphs(g, rows['local_idx'].values[pick])
    mirr, viol = [], []
    for d in orig:
        N, pat, seed = graph_arrays(d)
        mp_, ms_ = pat[:, ::-1].copy(), seed[:, ::-1].copy()
        viol.append(bool(np.any(mp_[seed == 1] == 0)))     # hole where the generator forces metal
        mirr.append(build_pyg_graph(mp_, d.y.squeeze(0).numpy(), ms_, N))
    viol = np.array(viol)
    Po = predict_db(BEST, orig); Pm = predict_db(BEST, mirr)
    gap_cache = np.abs(Po - P[pick]).max()
    print(f"   re-prediction vs Chunk 28 cache: max |diff| {gap_cache:.2e} dB")
    assert gap_cache < 1e-3, "re-prediction does not match the cache: wrong checkpoint, stats or featurizer"
    Tt = T[pick]
    for name, msk in (('all', np.ones(200, bool)), ('in_support', ~viol), ('violates_support', viol)):
        if msk.sum() == 0:
            continue
        mse_o = np.mean((Po[msk] - Tt[msk]) ** 2)
        mse_avg = np.mean(((Po[msk] + Pm[msk]) / 2 - Tt[msk]) ** 2)
        mir_rows.append(dict(item='mirror', grid=g, split=split, subset=name, n=int(msk.sum()),
                             share_violating=viol.mean(), mse_orig_vs_mirror_pred=np.mean((Po[msk] - Pm[msk]) ** 2),
                             mse_before=mse_o, mse_after=mse_avg, reduction=mse_o - mse_avg,
                             reduction_pct=100 * (mse_o - mse_avg) / mse_o))
MIR = pd.DataFrame(mir_rows)
if len(MIR):
    print(MIR.round(4).to_string(index=False))
    ins = MIR[MIR['subset'] == 'in_support']
    gain = 100 * ins['reduction'].sum() / ins['mse_before'].sum() if len(ins) else 0.0
    AUG = bool(MIRROR_OK and gain > 3.0)
    print(f"\nPRE-REGISTERED: in-support mirror averaging gain {gain:.2f}% of MSE "
          f"({'>' if gain > 3 else '<='} 3%) -> mirror augmentation in Chunk 33: {'YES' if AUG else 'NO'}")
else:
    AUG = False
DECISIONS['mirror_augmentation'] = AUG

## Cell D3 — Seed ensemble bound
Runs the 5 seeds each of `ch22_L0` and κ = 1.5 at n = 4,000 on the 35×35 and 45×45 val graphs. It reports the curve-averaged ensemble MSE against the mean individual MSE. These arms' training pools cannot be shown to be 55-free, so results are labelled indicative.

**Writes:** `ch29_free_wins.csv` (clamping, mirror and ensemble rows).

In [ ]:
# ── Cell D3: ensemble bound (inference on existing seeds, grids 35/45 val) ──
rows_v, T_v, _ = curves_for('best_model', 'finetune_val', grids=(35, 45))
graphs_v = []
for g in (35, 45):
    graphs_v += load_graphs(g, rows_v[rows_v['grid_size'] == g]['local_idx'].values)
order_g = np.concatenate([np.full((rows_v['grid_size'] == g).sum(), g) for g in (35, 45)])
T_ord = np.concatenate([T_v[rows_v['grid_size'].values == g] for g in (35, 45)])
for i in (0, len(graphs_v) - 1):
    assert np.abs(graphs_v[i].y.squeeze(0).numpy() - T_ord[i]).max() < 1e-3, "graph/curve misalignment"

ARMS = {'ch22_L0': sorted(glob.glob(f"{CKPT}/**/ch22_L0_seed*.pt", recursive=True)),
        'kappa1.5_n4000': sorted(set(glob.glob(f"{CKPT}/**/*k1.5*n4000*seed*.pt", recursive=True)
                                     + glob.glob(f"{CKPT}/**/*kappa1.5*n4000*seed*.pt", recursive=True)))}
ens = []
for arm, paths in ARMS.items():
    print(f"{arm}: {len(paths)} checkpoints", [os.path.basename(p) for p in paths])
    if len(paths) < 2:
        print("   fewer than 2 seeds found; skipped"); continue
    preds, prov_flags = [], []
    for p in paths:
        m, ck = load_gnn(p)
        txt = json.dumps({k: str(v)[:200] for k, v in ck.items() if k not in ('model_state', 'model_state_dict',
                          'state_dict', 'optimizer_state', 'optimizer_state_dict')})
        prov_flags.append('55' in txt and ('grid' in txt.lower() or 'pool' in txt.lower()))
        preds.append(predict_db(m, graphs_v)); del m
    Pst = np.stack(preds)
    status = ('pool includes 55 per provenance - INDICATIVE ONLY' if any(prov_flags)
              else 'training-pool grids not determinable from provenance - INDICATIVE ONLY')
    for g in (35, 45):
        msk = order_g == g
        ind = np.mean([np.mean((Pst[s][msk] - T_ord[msk]) ** 2) for s in range(len(Pst))])
        en = np.mean((Pst[:, msk].mean(0) - T_ord[msk]) ** 2)
        ens.append(dict(item='ensemble', arm=arm, grid=g, n_seeds=len(Pst), mse_before=ind, mse_after=en,
                        reduction=ind - en, reduction_pct=100 * (ind - en) / ind, status=status))
ENS = pd.DataFrame(ens)
if len(ENS):
    print(ENS.round(4).to_string(index=False))
FREE = pd.concat([FW, MIR, ENS], ignore_index=True)
FREE.to_csv(out_path('ch29_free_wins.csv'), index=False)

## Cell E (1/2) — Laplacian spectra
**What it does:** for 500 graphs per grid, extracts the feed-connected 4-connected metal subgraph. Subgraphs with ≤ 21 nodes get `eig_valid = 0` and zero eigenvalues; the cell reports their count and functioning rate.

It computes the 20 smallest nonzero eigenvalues (`eigsh`, shift-invert) in three variants:

| variant | definition | role |
|---|---|---|
| V1 | normalized Laplacian, raw | control; must fail comparability, because λₖ ∝ (π/N)² |
| V2 | normalized × N² | rescaled |
| V3 | combinatorial / h² (mm⁻²) | continuum Neumann k², the physical one |

It also computes the connected and total metal areas in mm².

In [ ]:
# ── Cell E (1/2): feed-connected metal subgraph, Laplacian spectra in three variants ──
from scipy.sparse import coo_matrix, diags, identity
from scipy.sparse.linalg import eigsh
M_EIG = 20
STRUCT4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])

def feed_component(pattern, N):
    r0, c0 = feed_rc(N)
    lab, _ = ndimage.label(pattern, structure=STRUCT4)
    cid = lab[r0, c0]
    return (lab == cid) if cid > 0 else np.zeros_like(pattern, bool)

def laplacians(mask):
    idx = -np.ones(mask.shape, int); idx[mask] = np.arange(mask.sum())
    a = (mask[:, :-1] & mask[:, 1:]); b = (mask[:-1, :] & mask[1:, :])
    i1, i2 = idx[:, :-1][a], idx[:, 1:][a]
    j1, j2 = idx[:-1, :][b], idx[1:, :][b]
    src = np.concatenate([i1, i2, j1, j2]); dst = np.concatenate([i2, i1, j2, j1])
    n = int(mask.sum())
    A = coo_matrix((np.ones(len(src)), (src, dst)), shape=(n, n)).tocsr()
    deg = np.asarray(A.sum(axis=1)).ravel()
    Lc = diags(deg) - A
    dinv = diags(1.0 / np.sqrt(deg))
    Ln = identity(n) - dinv @ A @ dinv
    return Lc.tocsc(), Ln.tocsc()

def smallest_nonzero(L, m=M_EIG):
    vals = eigsh(L, k=m + 1, sigma=-1e-3, which='LM', tol=1e-9, return_eigenvectors=False)
    vals = np.sort(vals)
    assert abs(vals[0]) < 1e-6, f"smallest eigenvalue {vals[0]:.3g} is not zero (component not connected?)"
    return vals[1:]

print(f"eigsh: shift-invert at sigma = -1e-3, tol = 1e-9, m = {M_EIG} smallest nonzero eigenvalues")
rngE = np.random.RandomState(295)
E_rows, E_graph_cache = [], {}
for split, g in (('25x25_test', 25), ('finetune_val', 35), ('finetune_val', 45)):
    rows, T, P = curves_for('best_model', split, grids=(g,))
    pick = np.sort(rngE.choice(len(rows), 500, replace=False))
    gs = load_graphs(g, rows['local_idx'].values[pick])
    E_graph_cache[g] = (rows.iloc[pick].reset_index(drop=True), T[pick], P[pick], gs)
    h = PIXEL_SIZE_MM[g]
    for k, d in enumerate(gs):
        N, pat, _ = graph_arrays(d)
        assert N == g
        t0 = time.perf_counter()
        comp = feed_component(pat.astype(bool), N)
        n_nodes = int(comp.sum())
        rec = dict(grid=g, local_idx=int(rows['local_idx'].values[pick][k]), n_nodes=n_nodes,
                   area_conn_mm2=n_nodes * h * h, area_metal_mm2=float(pat.sum()) * h * h,
                   label_min=bool(T[pick][k].min() < -10.0), f_res=float(F_GHZ[np.argmin(T[pick][k])]),
                   true_min_db=float(T[pick][k].min()))
        if n_nodes <= M_EIG + 1:
            rec['eig_valid'] = 0
            for v in ('V1', 'V2', 'V3'):
                rec.update({f'{v}_{i}': 0.0 for i in range(M_EIG)})
        else:
            rec['eig_valid'] = 1
            Lc, Ln = laplacians(comp)
            ln, lc = smallest_nonzero(Ln), smallest_nonzero(Lc)
            for v, arr in (('V1', ln), ('V2', ln * N * N), ('V3', lc / (h * h))):
                rec.update({f'{v}_{i}': float(arr[i]) for i in range(M_EIG)})
        rec['time_s'] = time.perf_counter() - t0
        E_rows.append(rec)
EIG = pd.DataFrame(E_rows)
print(EIG.groupby('grid').agg(n=('n_nodes', 'size'), invalid=('eig_valid', lambda s: int((s == 0).sum())),
                             median_time_ms=('time_s', lambda s: 1e3 * s.median())).to_string())
inv = EIG[EIG['eig_valid'] == 0]
print("LABEL_MIN functioning rate among eig_valid = 0 (should be near 0%):",
      inv.groupby('grid')['label_min'].mean().mul(100).round(1).to_dict())
print(f"Excluded from spectral statistics: {len(inv)} graphs")

## Cell E (2/2) — Comparability, physics, probes, selection
**What it does:**
- Extracts `best_model`'s 256-d readout with a forward hook, and proves the hooked pass reproduces the cached predictions.
- **Comparability:** a variant is COMPARABLE if the median λₖ ratios 45/25 and 35/25 both lie in [0.8, 1.25] for ≥ 15 of 20 eigenvalues.
- **Physics check:** Spearman ρ of √λ₁ against true f_res per grid.
- **Cross-scale probes:** gradient-boosted probes trained on 25×25 and tested on 45×45, from eigenvalues versus from the readout.

**Pre-registered selection:** carry forward the COMPARABLE V2/V3 variant with the best f_res R². Stop if V1 passes; drop the encoder if neither V2 nor V3 passes.

**Writes:** `ch29_spectral_viability.csv`, `ch29_eigs_sample.parquet`, `figures/ch29_spectral_viability.png`.

In [ ]:
# ── Cell E (2/2): comparability, physics check, probes, pre-registered selection ──
from sklearn.ensemble import HistGradientBoostingRegressor
print("""PRE-REGISTERED SELECTION: carry forward the COMPARABLE variant (V2 or V3) with the highest
train-25 / test-45 probe R^2 on f_res. COMPARABLE = median ratios lambda_k(45)/lambda_k(25) AND
lambda_k(35)/lambda_k(25) both in [0.8, 1.25] for >= 15 of 20 eigenvalues. If V1 passes, STOP.
If neither V2 nor V3 is COMPARABLE: SPECTRAL ENCODER DROPPED.
""")
V = EIG[EIG['eig_valid'] == 1]
spec_rows = []

# readout embeddings, with proof that the hooked run reproduces the cached predictions
READ = {}
for g, (rows, T, P, gs) in E_graph_cache.items():
    Pn, R = predict_db(BEST, gs, want_readout=True)
    gap = np.abs(Pn - P).max()
    print(f"{g}x{g}: hooked best_model vs Chunk 28 cache, max |diff| = {gap:.2e} dB")
    assert gap < 1e-3, "hooked forward pass does not reproduce the cache"
    assert R.shape == (len(gs), 256), f"readout shape {R.shape}"
    READ[g] = pd.DataFrame(R, index=rows['local_idx'].values)

def probe_r2(Xtr, ytr, Xte, yte):
    m = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(Xtr, ytr)
    return r2_score(yte, m.predict(Xte))

def feats(sub, v):
    return np.column_stack([sub[[f'{v}_{i}' for i in range(M_EIG)]].values,
                            sub[['area_conn_mm2', 'area_metal_mm2', 'eig_valid']].values])

result = {}
for v in ('V1', 'V2', 'V3'):
    med = V.groupby('grid')[[f'{v}_{i}' for i in range(M_EIG)]].median()
    r45, r35 = (med.loc[45] / med.loc[25]).values, (med.loc[35] / med.loc[25]).values
    ok = ((r45 >= 0.8) & (r45 <= 1.25) & (r35 >= 0.8) & (r35 <= 1.25)).sum()
    comparable = ok >= 15
    rhos = {}
    for g in GRIDS:
        s = V[(V['grid'] == g) & V['label_min']]
        rhos[g] = spearmanr(np.sqrt(s[f'{v}_0']), s['f_res'])[0] if len(s) > 10 else np.nan
    tr, te = EIG[EIG['grid'] == 25], EIG[EIG['grid'] == 45]
    trf, tef = tr[tr['label_min']], te[te['label_min']]
    r2_f = probe_r2(feats(trf, v), trf['f_res'], feats(tef, v), tef['f_res'])
    r2_d = probe_r2(feats(tr, v), tr['true_min_db'], feats(te, v), te['true_min_db'])
    result[v] = dict(comparable=comparable, r2_fres=r2_f)
    print(f"{v}: ratio 45/25 (k=1..5) {np.round(r45[:5], 3)}, 35/25 {np.round(r35[:5], 3)}; "
          f"{ok}/20 in band -> {'COMPARABLE' if comparable else 'NOT comparable'}")
    print(f"    Spearman sqrt(lambda_1) vs f_res: " + ", ".join(f"{g}: {rhos[g]:+.3f}" for g in GRIDS))
    print(f"    probe train-25/test-45: R^2 f_res {r2_f:+.3f}, R^2 min dB {r2_d:+.3f}")
    for i in range(M_EIG):
        spec_rows.append(dict(variant=v, k=i + 1, ratio_45_25=r45[i], ratio_35_25=r35[i],
                              comparable=comparable, rho_25=rhos[25], rho_35=rhos[35], rho_45=rhos[45],
                              r2_fres_25to45=r2_f, r2_mindb_25to45=r2_d))

# same probe on the learned 256-d readout
tr, te = EIG[EIG['grid'] == 25], EIG[EIG['grid'] == 45]
Rtr, Rte = READ[25].loc[tr['local_idx']].values, READ[45].loc[te['local_idx']].values
ftr, fte = tr['label_min'].values, te['label_min'].values
r2_read_f = probe_r2(Rtr[ftr], tr['f_res'][ftr], Rte[fte], te['f_res'][fte])
r2_read_d = probe_r2(Rtr, tr['true_min_db'], Rte, te['true_min_db'])
print(f"\nREADOUT (best_model 256-d) probe train-25/test-45: R^2 f_res {r2_read_f:+.3f}, R^2 min dB {r2_read_d:+.3f}")

assert not result['V1']['comparable'], ("STOP: V1 (raw normalized) PASSED comparability, which "
                                        "contradicts the scaling argument - inspect the subgraph extraction")
cands = [v for v in ('V2', 'V3') if result[v]['comparable']]
CHOSEN = max(cands, key=lambda v: result[v]['r2_fres']) if cands else None
if CHOSEN is None:
    print("\nSPECTRAL ENCODER DROPPED")
else:
    print(f"\nCHOSEN VARIANT: {CHOSEN} (R^2 f_res {result[CHOSEN]['r2_fres']:+.3f} vs readout {r2_read_f:+.3f})")
    if result[CHOSEN]['r2_fres'] > r2_read_f:
        print("   Eigenvalues beat the learned embedding on cross-scale f_res: the most actionable finding here.")
SPEC = pd.DataFrame(spec_rows)
SPEC['chosen'] = SPEC['variant'] == CHOSEN
SPEC['r2_fres_readout'] = r2_read_f; SPEC['r2_mindb_readout'] = r2_read_d
SPEC.to_csv(out_path('ch29_spectral_viability.csv'), index=False)
EIG.to_parquet(out_path('ch29_eigs_sample.parquet'), index=False)
DECISIONS['spectral_variant'] = CHOSEN or 'DROPPED'

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for ax, v in zip(axs, ('V1', 'V2', 'V3')):
    for g in GRIDS:
        ax.plot(range(1, M_EIG + 1), V[V['grid'] == g][[f'{v}_{i}' for i in range(M_EIG)]].median().values,
                'o-', ms=3, label=f'{g}x{g}')
    ax.set_yscale('log'); ax.set_xlabel('eigenvalue index k'); ax.set_ylabel('median lambda_k')
    ax.set_title(f"{v}{' (chosen)' if v == CHOSEN else ''}"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(out_path('ch29_spectral_viability.png', 'fig'), dpi=130); plt.show()

## Cell F — Guards
**What it does:**
- Asserts the parquet counts before filtering.
- Asserts there are no grid-55 rows in any `ch29_` table.
- Checks every bin assignment satisfies `lo < min_db ≤ hi`.
- Confirms no `ch28_*` artifact was modified.
- Confirms every new file starts with `ch29_`.
- Checks the resonator parameters: no NaN, and f0 in [0.8, 4.2] GHz.
- Confirms the Cell E variant and the label decision are recorded.

It writes `ch29_decisions.json`, the machine-readable input for Chunk 30, and states that no training or model selection occurred.

In [ ]:
# ── Cell F: guards ──
fails = []
def guard(ok, msg):
    print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")
    if not ok: fails.append(msg)

guard(N_ROWS_PRE == 1_125_040 and N_MODELS_PRE == 49, "parquet row/model counts before the grid-55 filter")
for p in WRITTEN:
    if not os.path.exists(p) or not p.endswith(('.csv', '.parquet')):
        continue
    t = pd.read_csv(p) if p.endswith('.csv') else pd.read_parquet(p)
    for col in ('grid', 'grid_size'):
        if col in t.columns:
            guard(not (t[col] == 55).any(), f"no grid-55 row in {os.path.basename(p)}")
_b = S[S['label_min']]
_mins = Y[_b['curve_id'].values].min(axis=1)
_ok = all(lo < v <= hi for v, lbl in zip(_mins, _b['depth_bin']) for lo, hi, l in DEPTH_BINS if l == lbl)
guard(_ok, "every depth-bin assignment satisfies lo < min_db <= hi")
_now = {p: os.path.getmtime(p) for p in CH28_MTIMES}
guard(all(_now[p] == CH28_MTIMES[p] for p in CH28_MTIMES), f"no ch28_* artifact modified ({len(CH28_MTIMES)} checked)")
guard(all(os.path.basename(p).startswith('ch29_') for p in WRITTEN), f"all {len(WRITTEN)} new files start with ch29_")
if DECISIONS.get('cell_a_branch') != 'DEAD':
    _par = pd.read_parquet(f"{ART}/ch29_resonator_params_sample.parquet")
    guard(not _par[['f0_GHz', 'Q', 'rmin']].isna().any().any(), "resonator params: no NaN in f0/Q/rmin")
    guard(_par['f0_GHz'].between(0.8, 4.2).all(), "resonator params: every f0 in [0.8, 4.2] GHz")
guard('spectral_variant' in DECISIONS, f"Cell E variant recorded: {DECISIONS.get('spectral_variant')}")
guard(DECISIONS.get('label') == 'LABEL_MIN', "Cell G label decision recorded: LABEL_MIN")

with open(out_path('ch29_decisions.json'), 'w') as fp:
    json.dump({k: (v.item() if hasattr(v, 'item') else v) for k, v in DECISIONS.items()}, fp, indent=2)
print("\nDECISIONS:", json.dumps({k: (v.item() if hasattr(v, 'item') else v) for k, v in DECISIONS.items()}, indent=2))
print("\nNo training, no optimizer step and no model selection occurred in this notebook.")
print("Forward passes: Cell D (best_model, ch22_L0 and kappa1.5 seeds) and Cell E (best_model readout),")
print("all eval mode under torch.no_grad(). Test splits touched: 25x25_test (diagnostics only).")
assert not fails, f"{len(fails)} guard(s) failed: {fails}"
print("ALL GUARDS PASSED")